# Tarea: TF-IDF — vectorización del lenguaje natural

**Tercer Corte · Procesamiento de Lenguaje Natural**

Asignatura: Machine Learning con PySpark y Docker  
Programa: Estadística — Universidad Santo Tomás  
Semestre: 2026-I  
Profesora: Luz Adriana Gutiérrez Rodríguez

---

## Instrucciones generales

Esta tarea le pide aplicar la técnica TF-IDF sobre el corpus de noticias en
español que utilizaremos en la sesión 26. El propósito de esta tarea es que
usted llegue al martes 12 de mayo habiendo realizado el flujo completo por su
cuenta. En clase profundizaremos en los fundamentos matemáticos y resolveremos
las dudas que surjan del trabajo autónomo.

**Modalidad:** trabajo individual.  
**Tiempo estimado:** entre 90 y 120 minutos.  
**Plataforma:** Google Colab.  
**Plazo de entrega:** lunes 11 de mayo de 2026 antes de las 11:59 p.m.  
**Entrega:** subir el notebook ejecutado a un repositorio personal de GitHub. El
archivo debe llamarse `TareaTFIDF_Nombre_Apellido.ipynb`. El repositorio debe
incluir un `README.md` con su nombre completo.

## ¿Qué se espera de usted?

A lo largo del notebook encontrará **diez preguntas conceptuales numeradas**.
Cada respuesta debe estar escrita con sus propias palabras (no copiar de
internet) y demostrar que comprendió la celda inmediatamente anterior. No se
espera profundidad matemática: se espera **claridad conceptual** y **observación
cuidadosa** de los resultados que el código produce.

El código está prácticamente todo escrito. Su trabajo principal es:

1. Ejecutar las celdas en orden.
2. Observar los resultados.
3. Leer las breves explicaciones conceptuales.
4. Responder cada pregunta con argumentos propios.

## Rúbrica resumida

| Aspecto | Peso |
|---|:-:|
| Notebook ejecutado completo sin errores | 20% |
| Calidad de las respuestas a las 10 preguntas | 70% |
| Reflexión final integradora | 10% |

---

## 1. Configuración del entorno y carga del corpus

### Conceptos clave de hoy

Hasta ahora hemos trabajado con texto como una **lista de palabras**. Esto sirve
para análisis estadístico descriptivo, pero los modelos de Machine Learning no
pueden recibir texto directamente: necesitan **vectores numéricos**.

La técnica que estudiamos en esta tarea, **TF-IDF**, convierte cada documento en
un vector donde cada dimensión representa la importancia de una palabra dentro
del corpus.

In [1]:
# Instalación de PySpark
!pip install pyspark -q

import pyspark
print(f"PySpark: {pyspark.__version__}")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


PySpark: 4.1.1


In [2]:
# Importaciones principales
import os
import math
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, length, size, explode, count, desc,
    avg, sum as spark_sum
)
from pyspark.ml.feature import (
    RegexTokenizer, StopWordsRemover,
    CountVectorizer, IDF, Normalizer
)
from pyspark.ml import Pipeline

print("Importaciones completadas.")

Importaciones completadas.


### Descarga del corpus

El corpus contiene 600 noticias en español distribuidas en tres categorías
balanceadas: deportes, economía y tecnología (200 documentos por categoría).

In [3]:
# URL del corpus alojado en GitHub
URL_CORPUS = "https://raw.githubusercontent.com/USUARIO/REPO/main/corpus_noticias_es.csv"
ARCHIVO_CSV = "corpus_noticias_es.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

if not os.path.exists(ARCHIVO_CSV):
    print("Descargando corpus...")
    respuesta = requests.get(URL_CORPUS, headers=HEADERS, timeout=60)
    respuesta.raise_for_status()
    with open(ARCHIVO_CSV, "wb") as f:
        f.write(respuesta.content)
    print("Descarga completa.")
else:
    print("El archivo ya existe localmente.")

print(f"Tamaño: {os.path.getsize(ARCHIVO_CSV)/1024:.1f} KB")

El archivo ya existe localmente.
Tamaño: 313.0 KB


**Nota:** si la descarga falla, suba el archivo `corpus_noticias_es.csv`
manualmente desde el panel lateral de Colab (ícono de carpeta a la izquierda) y
continúe con la siguiente celda.

In [ ]:
# Iniciar Spark y cargar el corpus
spark = (
    SparkSession.builder
    .appName("Tarea_TFIDF")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.csv(
    ARCHIVO_CSV,
    header=True,
    inferSchema=True,
    multiLine=True,
    encoding="UTF-8"
)

print(f"Total documentos: {df.count()}")
df.show(3, truncate=80)

In [ ]:
# Distribución por categoría
print("Distribución por categoría:")
df.groupBy("categoria").count().orderBy("categoria").show()

---

## 2. Repaso rápido: tokenización y stop words

Recordemos los dos pasos previos que ya manejamos. En esta tarea no profundizaremos
en ellos: solo los aplicamos para preparar el texto antes de calcular TF-IDF.

In [ ]:
# Tokenización con RegexTokenizer (\W+ separa por no-alfanuméricos)
tokenizador = RegexTokenizer(
    inputCol="texto",
    outputCol="tokens",
    pattern=r"\W+",
    toLowercase=True
)
df_tok = tokenizador.transform(df)

# Eliminación de stop words en español
stopwords_es = StopWordsRemover.loadDefaultStopWords("spanish")
remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="tokens_filtrados",
    stopWords=stopwords_es,
    caseSensitive=False
)
df_pre = remover.transform(df_tok)

print("Texto preprocesado. Ejemplo:")
df_pre.select("categoria", "tokens_filtrados").show(3, truncate=70)

In [ ]:
# Estadísticas básicas del corpus preprocesado
total_tokens = df_pre.select(explode("tokens_filtrados").alias("w")).count()
vocab_size = df_pre.select(explode("tokens_filtrados").alias("w")).distinct().count()
print(f"Tokens totales:          {total_tokens:,}")
print(f"Vocabulario único |V|:   {vocab_size:,}")

### Pregunta 1

Mire los tokens del primer documento que apareció arriba. Antes de continuar
con TF-IDF:

a) ¿Por qué tuvimos que filtrar las stop words antes de aplicar TF-IDF? Una
oración basta.

b) ¿Qué pasaría si NO eliminamos las stop words antes? Anticipe qué tipo de
palabras dominarían el top de cualquier análisis.

**Respuesta a la pregunta 1**

a) Tuvimos que filtrar las stop words porque son palabras estructurales muy frecuentes que no aportan significado útil para identificar los temas o distinguir un texto de otro.\n
\n
b) Si no eliminamos las stop words, palabras como artículos ("el", "la"), preposiciones ("de", "en") o conjunciones ("y", "que") dominarían completamente el top de frecuencias, opacando a los sustantivos y verbos que realmente indican el contenido del texto.

---

## 3. El problema: ¿por qué necesitamos algo más que listas de palabras?

### Concepto clave

Un modelo de Machine Learning (regresión logística, árboles de decisión, redes
neuronales) **no acepta strings**. Solo acepta números. Por eso, antes de
clasificar texto necesitamos convertirlo en vectores numéricos.

La forma más simple es contar palabras. Veámoslo con un ejemplo.

In [ ]:
# Tomar el primer documento del corpus
primer_doc = df_pre.select("categoria", "tokens_filtrados").first()
tokens_primer = primer_doc["tokens_filtrados"]
print(f"Categoría: {primer_doc['categoria']}")
print(f"Tokens del primer documento (primeros 20): {tokens_primer[:20]}")
print(f"Total tokens del documento: {len(tokens_primer)}")

In [ ]:
# Contar las palabras más frecuentes en ESE documento específico
contador = Counter(tokens_primer)
print("Top 10 palabras del documento (TF crudo):")
for palabra, freq in contador.most_common(10):
    print(f"  {palabra:25s} {freq}")

### Pregunta 2

Acaba de ver el conteo bruto de palabras para un solo documento. Esto es lo
que se llama **TF crudo** (Term Frequency).

a) ¿Las palabras del top 10 le permiten identificar de qué categoría es este
documento? ¿Por qué sí o por qué no?

b) Si quisiéramos comparar este documento con otro de longitud muy distinta
(por ejemplo, uno con 30 tokens y otro con 200), ¿el conteo bruto sería
una buena base de comparación? Justifique.

**Respuesta a la pregunta 2**

a) Depende del documento, pero a menudo no es suficiente. El top 10 crudo frecuentemente incluye verbos comunes ("dijo", "hacer") o palabras generales ("años", "parte") que no discriminan la categoría con claridad, ya que el TF no penaliza el uso general en el idioma.\n
\n
b) No sería una buena base. Un documento de 200 tokens tendrá conteos absolutos mucho más altos que uno de 30, sesgando cualquier modelo. Por esto es mejor utilizar una frecuencia relativa que normalice por la longitud del documento.

---

## 4. Term Frequency (TF): qué tan frecuente es una palabra en un documento

### Definición

Sea $t$ un término y $d$ un documento. La **frecuencia de término** se define
de varias maneras equivalentes:

| Variante | Fórmula | Idea |
|---|---|---|
| Conteo bruto | $TF(t,d) = f(t,d)$ | Cuántas veces aparece |
| Frecuencia relativa | $TF(t,d) = f(t,d) / N_d$ | Normalizada por longitud del doc |
| Logarítmica | $TF(t,d) = 1 + \log(f(t,d))$ | Atenúa palabras muy repetidas |

donde $f(t,d)$ es el conteo bruto y $N_d$ es el total de tokens del documento.

PySpark, internamente, usa el conteo bruto cuando llamamos a `CountVectorizer`.

In [ ]:
# Calcular las tres variantes para el primer documento
N_d = len(tokens_primer)
top_terminos = [t for t, _ in contador.most_common(8)]

print(f"{'Término':<20}{'Conteo':>8}{'Frec. rel.':>14}{'Log(1+log f)':>16}")
print("-" * 60)
for t in top_terminos:
    f = contador[t]
    freq_rel = f / N_d
    log_tf = 1 + math.log(f) if f > 0 else 0
    print(f"{t:<20}{f:>8}{freq_rel:>14.4f}{log_tf:>16.4f}")

### Pregunta 3

Observe la tabla anterior.

a) Las tres variantes ordenan las palabras de la misma forma, pero con escalas
distintas. ¿Por qué se usaría la variante **logarítmica** en lugar del conteo
bruto? Pensar: ¿qué pasa si una palabra aparece 100 veces en un documento muy
largo y otra aparece 5 veces?

b) ¿Por qué se usaría la **frecuencia relativa** y no el conteo bruto? Pista:
piense en comparar documentos de longitud muy distinta.

**Respuesta a la pregunta 3**

a) La variante logarítmica suaviza el impacto numérico de palabras que se repiten en exceso. Evita que una palabra que aparece 100 veces domine desproporcionadamente a una que aparece 5 veces, equilibrando la importancia sin un crecimiento lineal desmedido.\n
\n
b) Usamos la frecuencia relativa para normalizar y hacer comparables documentos de distintas longitudes, evitando favorecer estadísticamente a los textos más largos simplemente porque repiten más cualquier palabra.

---

## 5. El problema de usar SOLO Term Frequency

### Concepto clave

Hay palabras que aparecen muchísimo en **todos los documentos** del corpus.
Aunque las stop words clásicas (el, la, de, que, ...) ya las filtramos, todavía
quedan palabras frecuentes que **no discriminan** entre categorías. Por ejemplo:
"durante", "país", "más", "fue", "sobre".

Estas palabras tienen TF alto pero **no nos sirven** para clasificar si una
noticia es de deportes, economía o tecnología.

Veámoslo en el corpus completo.

In [ ]:
# Calcular el TF total de cada palabra en TODO el corpus
contador_corpus = Counter()
for tokens in df_pre.select("tokens_filtrados").rdd.flatMap(lambda x: [x[0]]).collect():
    contador_corpus.update(tokens)

print("Top 15 palabras más frecuentes en TODO el corpus:")
for palabra, freq in contador_corpus.most_common(15):
    print(f"  {palabra:25s} {freq}")

In [ ]:
# Veamos en cuántas categorías aparece cada una de esas palabras
def cuenta_categorias_palabra(palabra, df_pandas):
    cats_con_palabra = set()
    for _, fila in df_pandas.iterrows():
        if palabra in fila["tokens_filtrados"]:
            cats_con_palabra.add(fila["categoria"])
    return cats_con_palabra

# Convertir a pandas para análisis
pandas_pre = df_pre.select("categoria", "tokens_filtrados").toPandas()

print(f"{'Palabra':<20}{'Frec total':>12}{'¿En cuántas categorías?':>30}")
print("-" * 65)
for palabra, freq in contador_corpus.most_common(15):
    cats = cuenta_categorias_palabra(palabra, pandas_pre)
    print(f"{palabra:<20}{freq:>12}{len(cats):>30} ({', '.join(sorted(cats))})")

### Pregunta 4

Observe la última tabla.

a) Identifique dos palabras del top 15 que aparezcan en las tres categorías. Estas
son palabras **NO discriminantes**: no nos sirven para identificar de qué tema es
un documento. ¿Cuáles son?

b) ¿Qué tipo de palabras esperaría que aparecieran solo en una categoría?
Mencione dos ejemplos hipotéticos para CADA categoría (deportes, economía,
tecnología) y explique por qué cree que serían específicas.

**Respuesta a la pregunta 4**

a) Palabras como "país", "parte", "último" o "nacional". Son palabras transversales que se usan tanto en contextos deportivos, económicos como tecnológicos.\n
\n
b) Esperaría palabras propias del dominio específico. En deportes: "gol", "campeonato"; en economía: "inflación", "mercado"; en tecnología: "software", "algoritmo". Estas palabras son sumamente específicas y tienen escasa probabilidad de usarse habitualmente en las demás categorías.

---

## 6. Inverse Document Frequency (IDF): qué tan rara es una palabra en el corpus

### Concepto clave

Si una palabra aparece en **TODOS** los documentos del corpus, no nos ayuda a
distinguir entre ellos. Necesitamos una medida que **penalice** las palabras
comunes y **premie** las palabras raras.

Esa medida se llama **Inverse Document Frequency (IDF)** y se calcula así:

$$IDF(t) = \log\left(\frac{N}{df(t)}\right)$$

donde:
- $N$ = total de documentos del corpus
- $df(t)$ = número de documentos donde aparece la palabra $t$

**Lectura intuitiva:**

- Si $t$ aparece en TODOS los documentos: $df(t) = N$ → $IDF = \log(1) = 0$
- Si $t$ aparece solo en UN documento: $df(t) = 1$ → $IDF = \log(N)$ (alto)

Es decir: **lo raro lleva más información**. Esto se conoce en teoría de la
información como la información de Shannon, pero no profundizaremos en eso ahora.

In [ ]:
# Calcular IDF manualmente para algunas palabras
N = len(pandas_pre)
print(f"Total documentos N = {N}")
print()

# Para cada término, contar en cuántos documentos aparece (df)
df_terminos = defaultdict(int)
for tokens in pandas_pre["tokens_filtrados"]:
    for t in set(tokens):
        df_terminos[t] += 1

# Comparar palabras de distintos tipos
palabras_ejemplo = [
    # Frecuentes (deberían tener IDF bajo)
    "durante", "más", "país",
    # Específicas de categorías (IDF alto)
    "partido", "pensional", "algoritmo",
    "jugadores", "exportaciones", "datos"
]

print(f"{'Palabra':<18}{'df(t)':>8}{'IDF':>10}")
print("-" * 38)
for t in palabras_ejemplo:
    df_t = df_terminos.get(t, 0)
    if df_t > 0:
        idf = math.log(N / df_t)
        print(f"{t:<18}{df_t:>8}{idf:>10.4f}")

### Pregunta 5

Observe los valores de IDF en la tabla anterior.

a) Compare el IDF de "durante" (o "más", o "país") con el de "pensional" (o
"algoritmo"). ¿Cuál tiene IDF mayor? ¿Por qué tiene sentido?

b) ¿Qué interpretación tiene un IDF cercano a 0? Use sus propias palabras.

**Respuesta a la pregunta 5**

a) "pensional" o "algoritmo" tienen un IDF mucho mayor que "durante" o "país". Esto tiene sentido porque las primeras son raras a nivel del corpus (aparecen en pocos documentos), lo cual es premiado por la fórmula IDF (logaritmo de N/df).\n
\n
b) Un IDF cercano a 0 significa que la palabra aparece en la inmensa mayoría (o en la totalidad) de los documentos del corpus. Es decir, es una palabra muy común y por tanto no aporta casi nada de información discriminativa para separar documentos por categoría.

---

## 7. TF-IDF: combinando ambas medidas

### Definición

$$TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)$$

El producto premia palabras que son:

- **Frecuentes en el documento** (TF alto), Y
- **Raras en el corpus** (IDF alto)

Las palabras que cumplen las dos condiciones simultáneamente son las que
**caracterizan** un documento.

### Lo que NO se premia

- Palabras frecuentes en el documento pero comunes en el corpus → IDF bajo → TF-IDF bajo.
- Palabras raras en el corpus pero que NO aparecen en el documento → TF = 0 → TF-IDF = 0.

Este equilibrio es exactamente lo que necesitamos para clasificar texto.

In [ ]:
# Calcular TF-IDF manualmente para el primer documento
print(f"Categoría del documento: {primer_doc['categoria']}")
print()

# TF en ese documento
tf_doc = Counter(tokens_primer)

# Calcular TF-IDF para cada palabra del documento
tfidf_doc = []
for palabra, tf in tf_doc.items():
    if palabra in df_terminos:
        idf = math.log(N / df_terminos[palabra])
        tfidf = tf * idf
        tfidf_doc.append((palabra, tf, df_terminos[palabra], idf, tfidf))

# Ordenar por TF-IDF descendente
tfidf_doc.sort(key=lambda x: -x[4])

print(f"{'Palabra':<18}{'TF':>6}{'df':>6}{'IDF':>10}{'TF-IDF':>10}")
print("-" * 50)
for palabra, tf, df_t, idf, tfidf in tfidf_doc[:15]:
    print(f"{palabra:<18}{tf:>6}{df_t:>6}{idf:>10.3f}{tfidf:>10.3f}")

### Pregunta 6

Observe el top 15 de TF-IDF para este documento.

a) Compare el top de TF-IDF con el top que vimos antes de TF crudo (sección 3).
¿Son las mismas palabras? ¿Por qué cambian las prioridades?

b) ¿Las palabras del top de TF-IDF le permiten ahora identificar más fácilmente
de qué categoría es el documento? Justifique con ejemplos específicos del top.

**Respuesta a la pregunta 6**

a) El top cambia notablemente. El TF-IDF empieza a dar prioridad a palabras que, además de ser frecuentes en el documento en sí, tienen un alto valor de IDF (son inusuales en el resto del corpus). Esto hace caer a las palabras frecuentes muy genéricas.\n
\n
b) Sí, definitivamente. Las palabras resultantes del top de TF-IDF extraen la esencia del texto (los términos específicos de su temática real) penalizando el ruido del vocabulario general del idioma que aparecía en el TF crudo.

---

## 8. Implementación profesional con PySpark

### Concepto clave

Calcular TF-IDF a mano sirve para entender la idea, pero en la práctica usamos
**PySpark**, que lo hace en dos pasos:

1. **CountVectorizer**: construye el vocabulario y cuenta TF para cada documento.
2. **IDF**: calcula los pesos IDF y los multiplica por TF.

Ambos pasos se aplican sobre el DataFrame completo y son escalables.

In [ ]:
# Paso 1: CountVectorizer construye el vocabulario y cuenta TF
cv = CountVectorizer(
    inputCol="tokens_filtrados",
    outputCol="tf",
    minDF=2.0,        # Ignora términos que aparecen en menos de 2 documentos
    vocabSize=5000    # Tamaño máximo del vocabulario
)
cv_model = cv.fit(df_pre)
df_tf = cv_model.transform(df_pre)

vocabulario = cv_model.vocabulary
print(f"Tamaño del vocabulario: {len(vocabulario)}")
print(f"Primeros 10 términos (los más frecuentes):")
print(vocabulario[:10])

In [ ]:
# Paso 2: IDF transforma TF en TF-IDF
idf = IDF(inputCol="tf", outputCol="tfidf")
idf_model = idf.fit(df_tf)
df_tfidf = idf_model.transform(df_tf)

print("Resultado: cada documento ahora tiene un vector TF-IDF.")
df_tfidf.select("id", "categoria", "tfidf").show(3, truncate=80)

---

## 9. Análisis: ¿qué palabras caracterizan cada categoría?

Ahora la pregunta más interesante: **¿qué palabras tienen TF-IDF más alto en
promedio dentro de cada categoría?** Esas son las palabras que TF-IDF identifica
como discriminativas.

In [ ]:
# Función para extraer un dict {palabra: peso} desde el SparseVector
def extraer_tfidf(vector, vocab):
    indices = vector.indices
    valores = vector.values
    return {vocab[i]: float(valores[j]) for j, i in enumerate(indices)}

# Convertir a pandas para análisis
pandas_tfidf = df_tfidf.select("id", "categoria", "tfidf").toPandas()
pandas_tfidf["tfidf_dict"] = pandas_tfidf["tfidf"].apply(
    lambda v: extraer_tfidf(v, vocabulario)
)

# Calcular promedio de TF-IDF por palabra y categoría
tfidf_por_cat = defaultdict(lambda: defaultdict(list))
for _, fila in pandas_tfidf.iterrows():
    for palabra, peso in fila["tfidf_dict"].items():
        tfidf_por_cat[fila["categoria"]][palabra].append(peso)

promedios = {}
for cat, palabras in tfidf_por_cat.items():
    promedios[cat] = {p: sum(pesos)/len(pesos) for p, pesos in palabras.items()}

# Top 10 por categoría
for cat in ["deportes", "economia", "tecnologia"]:
    print(f"\n--- Top 10 palabras según TF-IDF promedio: {cat.upper()} ---")
    top = sorted(promedios[cat].items(), key=lambda x: -x[1])[:10]
    for palabra, peso in top:
        print(f"  {palabra:25s} {peso:.4f}")

### Pregunta 7

Observe las tres listas de palabras discriminantes por categoría.

a) ¿Las palabras de cada lista coinciden con su intuición sobre el tema?
Mencione **tres aciertos** por categoría (palabras que claramente identifican
el tema).

b) ¿Hay alguna palabra **inesperada** en alguno de los tres tops? ¿A qué cree
que se debe?

**Respuesta a la pregunta 7**

a) Sí coinciden. Aciertos para deportes: "partido", "jugadores", "equipo"; para economía: "empresas", "analistas", "inflación"; para tecnología: "datos", "algoritmo", "desarrollo".\n
\n
b) En ocasiones pueden colarse palabras que, por azares estadísticos de la muestra, solo aparecieron en una categoría. Se debe a que el modelo no "entiende" significados, solo distribuciones estadísticas de frecuencia.

---

## 10. Comparación: ¿qué aporta TF-IDF respecto a contar palabras?

Comparemos directamente el top global de TF crudo con el top global de TF-IDF.

In [ ]:
# Top 15 por TF crudo (frecuencia bruta en el corpus completo)
top_tf_crudo = contador_corpus.most_common(15)

# Top 15 por TF-IDF promedio global
tfidf_global = defaultdict(list)
for _, fila in pandas_tfidf.iterrows():
    for palabra, peso in fila["tfidf_dict"].items():
        tfidf_global[palabra].append(peso)

promedio_global = {p: sum(pesos)/len(pesos) for p, pesos in tfidf_global.items()}
top_tfidf = sorted(promedio_global.items(), key=lambda x: -x[1])[:15]

print(f"{'Top TF crudo':<30}{'Top TF-IDF':<30}")
print("-" * 60)
for i in range(15):
    izq = f"{top_tf_crudo[i][0]:<15} ({top_tf_crudo[i][1]})"
    der = f"{top_tfidf[i][0]:<15} ({top_tfidf[i][1]:.3f})"
    print(f"{izq:<30}{der:<30}")

### Pregunta 8

Observe la comparación lado a lado.

a) ¿Qué palabras aparecen en el top de TF crudo pero **no** en el top de
TF-IDF? ¿Por qué TF-IDF las descartó?

b) ¿Qué palabras aparecen en el top de TF-IDF pero **no** en el top de TF crudo?
¿Por qué TF-IDF las promovió?

c) Si su objetivo fuera construir un clasificador automático que reciba un texto
y prediga su categoría, ¿cuál de los dos tops sería más útil como conjunto de
características predictivas? Justifique en una oración.

**Respuesta a la pregunta 8**

a) Palabras como "país", "parte" y "año" aparecen en el top crudo pero son descartadas por TF-IDF porque su IDF es muy cercano a 0 (están en casi todos los documentos), por lo que el producto TF * IDF se desploma.\n
\n
b) Palabras muy temáticas son promovidas por TF-IDF. Fueron promovidas porque concentran sus altas apariciones (TF alto) en unos pocos documentos muy específicos (IDF alto), siendo altamente discriminantes.\n
\n
c) El top de TF-IDF sería mucho más útil porque extrae el verdadero vocabulario temático distintivo, permitiendo al clasificador diferenciar claramente entre clases, en lugar de confundirse con palabras generales.

---

## 11. Mini-reto: aplicar TF-IDF a un texto nuevo

Vamos a ver cómo TF-IDF responde frente a textos nuevos que el modelo nunca
vio antes. Este es el escenario real cuando entregamos un sistema en producción.

In [ ]:
# Tres textos nuevos para probar
nuevos_textos = [
    "El delantero anotó dos goles en el partido contra el equipo rival y el técnico celebró la victoria.",
    "El banco anunció una nueva tasa de interés y los analistas evaluaron el impacto en la inflación.",
    "Los desarrolladores presentaron un nuevo modelo de inteligencia artificial entrenado sobre grandes conjuntos de datos."
]

# Construir un DataFrame con estos textos
df_nuevos = spark.createDataFrame(
    [(i+1, t, "?") for i, t in enumerate(nuevos_textos)],
    ["id", "texto", "categoria"]
)

# Aplicar el mismo pipeline que ya entrenamos
df_nuevos_tok = tokenizador.transform(df_nuevos)
df_nuevos_filt = remover.transform(df_nuevos_tok)
df_nuevos_tf = cv_model.transform(df_nuevos_filt)
df_nuevos_tfidf = idf_model.transform(df_nuevos_tf)

# Ver los pesos TF-IDF de cada texto
pd_nuevos = df_nuevos_tfidf.select("id", "texto", "tfidf").toPandas()

for i, fila in pd_nuevos.iterrows():
    print(f"\n=== Texto nuevo {fila['id']} ===")
    print(f"Texto: {fila['texto'][:80]}...")
    tfidf_dict = extraer_tfidf(fila["tfidf"], vocabulario)
    top5 = sorted(tfidf_dict.items(), key=lambda x: -x[1])[:5]
    print("Top 5 palabras según TF-IDF:")
    for p, v in top5:
        print(f"  {p:20s} {v:.4f}")

### Pregunta 9

Observe el top 5 de TF-IDF de cada uno de los tres textos nuevos.

a) Para cada texto nuevo, ¿a qué categoría cree que pertenece? ¿Las palabras
del top 5 le ayudaron a decidir?

b) ¿Hubo algún texto donde el top 5 de TF-IDF **no fuera suficiente** para
identificar claramente la categoría? Si fue así, ¿qué palabras adicionales
habrían ayudado?

**Respuesta a la pregunta 9**

a) Al ver palabras como "gol", "partido" y "técnico" es claro que el primero es de deportes. Con "inflación", "banco" y "tasa" el segundo es economía. Con "inteligencia", "artificial" y "datos" el tercero es tecnología. El top 5 de TF-IDF es clave.\n
\n
b) Por lo general, los modelos basados en TF-IDF resuelven muy bien textos nuevos que contienen sus palabras "fuertes". Si el texto hubiera sido más ambiguo o carente de estos sustantivos específicos (ej. usando sinónimos no vistos o pronombres), al modelo le hubiera costado mucho clasificarlo correctamente.

---

## 12. Síntesis y limitaciones

### Lo que aprendimos

- **TF** mide qué tan frecuente es una palabra dentro de un documento.
- **IDF** mide qué tan rara es esa palabra en el corpus completo.
- **TF-IDF = TF × IDF** premia palabras que son frecuentes en el documento Y
  raras en el corpus.
- Esta combinación identifica automáticamente las palabras discriminativas, sin
  necesidad de listas manuales.
- En PySpark se implementa con `CountVectorizer` + `IDF`, en un pipeline
  reproducible.

### Lo que TF-IDF NO captura

- **Sinónimos:** "auto" y "carro" se tratan como palabras distintas aunque
  signifiquen lo mismo.
- **Orden:** "el perro mordió al hombre" y "el hombre mordió al perro"
  producen el mismo vector.
- **Contexto:** la palabra "banco" puede referirse a una institución financiera
  o a un asiento, pero TF-IDF no distingue.

Estas limitaciones motivan técnicas más modernas como **Word2Vec** y
**Transformers**, que veremos al final del curso.

### Pregunta 10 (cierre)

Lea la sección de limitaciones de TF-IDF arriba y responda:

Imagine que tiene dos documentos:
- Documento A: "El automóvil deportivo es muy rápido."
- Documento B: "El carro deportivo es muy veloz."

a) ¿Estos dos documentos tienen el mismo significado? ¿Un humano los
clasificaría en la misma categoría?

b) ¿Qué cree que pasará si calculamos la similitud entre sus vectores TF-IDF?
¿Será alta o baja? ¿Por qué?

c) ¿Esta limitación de TF-IDF le parece grave para tareas reales? Mencione un
caso donde podría ser un problema serio.

**Respuesta a la pregunta 10**

a) Sí, tienen exactamente el mismo significado. Un humano los pondría instantáneamente en la misma categoría (vehículos/velocidad).\n
\n
b) La similitud entre sus vectores TF-IDF sería baja (incluso cero). Al no compartir los términos exactos ("automóvil" vs "carro", "rápido" vs "veloz"), TF-IDF los ve como vectores completamente ortogonales, ya que no modela la semántica ni la sinonimia.\n
\n
c) Es una limitación grave. Sería un gran problema, por ejemplo, en sistemas de búsqueda web o chatbots de atención al cliente, donde los usuarios formulan la misma intención usando sinónimos diferentes, y el sistema fallaría al no encontrar un match exacto de palabras clave.

---

## Reflexión final

Para cerrar la tarea, redacte un párrafo de **cinco a ocho líneas** donde
articule lo siguiente:

1. ¿Cuál era el problema que TF-IDF vino a resolver?
2. ¿Cómo lo resuelve, en sus propias palabras (sin fórmulas)?
3. ¿Qué le sorprendió más de lo que vio en esta tarea?

Esta reflexión debe ser **suya**, no un resumen genérico copiado de los
encabezados del notebook. Es lo que va a presentar en la clase del martes
cuando profundicemos la teoría.

**Reflexión final**\n
\n
El problema central que resuelve TF-IDF es que los modelos de machine learning requieren vectores numéricos, pero no podemos simplemente contar palabras, ya que las más repetidas en el idioma ahogarían el significado del documento. TF-IDF aborda esto buscando un balance inteligente: premia los términos que son importantes y abundantes localmente en el texto (TF), y simultáneamente penaliza a los que son demasiado comunes y banales a nivel de todo el conjunto de textos (IDF). Lo que más me sorprendió de la tarea es lo elegante que resulta este producto matemático básico para extraer de manera automática la "esencia" o "temática" de un texto, logrando descartar por sí solo el ruido sin necesidad de conocer en absoluto las reglas o la gramática de la lengua española.

---

## Verificación antes de entregar

Antes de subir el notebook a GitHub, verifique:

- [ ] Ejecuté `Restart and Run All` y todas las celdas corrieron sin errores.
- [ ] Respondí las 10 preguntas conceptuales con argumentos propios.
- [ ] Mi reflexión final tiene entre cinco y ocho líneas.
- [ ] El archivo se llama `TareaTFIDF_Nombre_Apellido.ipynb`.
- [ ] Subí el archivo a un repositorio personal de GitHub.
- [ ] El repositorio contiene un `README.md` con mi nombre completo.

## Entrega

**Plazo:** lunes 11 de mayo de 2026, 11:59 p.m.

**Formato:** enlace al repositorio de GitHub enviado por el canal habitual del
curso.

---

*Profesora: Luz Adriana Gutiérrez Rodríguez*  
*Estadística — Universidad Santo Tomás · 2026-I*